# Actividad 2 · Interpretación y límites del análisis

**Recorrido elegido:** SQL + Python  
**Tiempo estimado:** 1 pomodoro (20 minutos)

> Realizá solamente uno de los dos recorridos equivalentes disponibles: **SQL + Python** o **SQL + R**.

En esta actividad vas a revisar cómo transformar un resultado numérico en una interpretación prudente. Trabajarás nuevamente con el dataset NSFG para distinguir entre **resultado**, **interpretación** y **conclusión**, y para reconocer los límites de la evidencia disponible.

## Preparación automática

Ejecutá la celda siguiente sin modificarla. Preparará el dataset, las variables necesarias y la conexión con DuckDB. Esta preparación técnica no forma parte de los contenidos evaluados.

In [ ]:
#@title Preparación automática del entorno (ejecutar sin modificar)
import importlib.util
import subprocess
import sys
from pathlib import Path
from urllib.request import urlretrieve

if importlib.util.find_spec("duckdb") is None:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "duckdb"])

import duckdb
import numpy as np
import pandas as pd

DATA_URL = "https://raw.githubusercontent.com/AllenDowney/ThinkStats/v3/data/2002FemPreg.dat.gz"
DATA_PATH = Path("2002FemPreg.dat.gz")
if not DATA_PATH.exists():
    urlretrieve(DATA_URL, DATA_PATH)

colspecs = [
    (0, 12), (12, 14), (56, 58), (58, 60), (274, 276),
    (276, 277), (277, 279), (283, 287), (422, 440),
]
names = [
    "caseid", "pregordr", "birthwgt_lb", "birthwgt_oz", "prglngth",
    "outcome", "birthord", "agepreg", "finalwgt",
]

embarazos = pd.read_fwf(
    DATA_PATH, names=names, colspecs=colspecs, compression="gzip"
)
embarazos["birthwgt_lb"] = embarazos["birthwgt_lb"].replace([51, 97, 98, 99], np.nan)
embarazos["birthwgt_oz"] = embarazos["birthwgt_oz"].replace([97, 98, 99], np.nan)
embarazos["totalwgt_lb"] = embarazos["birthwgt_lb"] + embarazos["birthwgt_oz"] / 16
embarazos["totalwgt_kg"] = (embarazos["totalwgt_lb"] * 0.45359237).round(3)
embarazos["agepreg"] = embarazos["agepreg"] / 100
embarazos = embarazos[[
    "caseid", "pregordr", "prglngth", "outcome", "birthord",
    "birthwgt_lb", "birthwgt_oz", "totalwgt_lb", "totalwgt_kg",
    "agepreg", "finalwgt",
]]

con = duckdb.connect(":memory:")
con.register("embarazos_df", embarazos)
con.execute("CREATE OR REPLACE TABLE embarazos AS SELECT * FROM embarazos_df")
print(f"Entorno preparado: {len(embarazos):,} registros disponibles en la tabla 'embarazos'.")

## Resultado de la comparación

Recuperaremos la comparación entre primeros bebés y otros. La consulta devuelve el tamaño de cada grupo, el promedio y el desvío estándar de la duración del embarazo.

In [ ]:
comparacion = con.sql("""SELECT
    CASE WHEN birthord = 1 THEN 'Primeros' ELSE 'Otros' END AS grupo,
    COUNT(*) AS cantidad,
    AVG(prglngth) AS promedio_semanas,
    STDDEV(prglngth) AS desvio_semanas
FROM embarazos
WHERE outcome = 1
GROUP BY grupo
ORDER BY grupo""").df()
comparacion

In [ ]:
promedio_primeros = comparacion.loc[
    comparacion["grupo"] == "Primeros", "promedio_semanas"
].iloc[0]
promedio_otros = comparacion.loc[
    comparacion["grupo"] == "Otros", "promedio_semanas"
].iloc[0]

diferencia_semanas = promedio_primeros - promedio_otros
diferencia_horas = diferencia_semanas * 7 * 24

print(f"Diferencia entre promedios: {diferencia_semanas:.4f} semanas")
print(f"Equivalencia aproximada: {diferencia_horas:.1f} horas")

El cálculo muestra un **resultado**. Para interpretarlo debemos compararlo con la variabilidad de los grupos y preguntarnos si la diferencia tiene relevancia práctica. Tampoco podemos atribuirle una causa únicamente a partir de esta comparación.

## Alcance de la muestra

La NSFG utilizada corresponde a Estados Unidos y reúne información de un período y una población determinados. Además, utiliza un diseño muestral en el que no todas las observaciones representan la misma cantidad de personas.

In [ ]:
con.sql("""SELECT
    COUNT(*) AS nacimientos_vivos,
    MIN(agepreg) AS edad_minima,
    MAX(agepreg) AS edad_maxima,
    MIN(finalwgt) AS peso_muestral_minimo,
    MAX(finalwgt) AS peso_muestral_maximo
FROM embarazos
WHERE outcome = 1""")

## Evaluación de afirmaciones

Utilizá los resultados anteriores para evaluar las siguientes afirmaciones. En cada caso indicá si está respaldada, parcialmente respaldada o no respaldada, y justificá tu decisión.

### Afirmación 1

> “Los primeros bebés llegan, en promedio, algunas horas más tarde que los demás.”

**Tu evaluación:** escribí acá tu respuesta e incorporá el tamaño de la diferencia y la variabilidad observada.

### Afirmación 2

> “Ser el primer bebé es la causa de una mayor duración del embarazo.”

**Tu evaluación:** escribí acá tu respuesta y explicá por qué una comparación exploratoria no permite demostrar causalidad.

### Afirmación 3

> “Los resultados pueden aplicarse sin restricciones a embarazos de cualquier país y período histórico.”

**Tu evaluación:** escribí acá tu respuesta considerando la procedencia y el alcance de la muestra.

## Promedio simple y promedio ponderado

La variable `finalwgt` es un peso muestral. Compararemos el promedio simple con un promedio que considera cuánto representa cada observación dentro de la población estudiada.

In [ ]:
con.sql("""SELECT
    'Sin ponderar' AS metodo,
    ROUND(AVG(prglngth), 3) AS duracion_promedio
FROM embarazos
WHERE outcome = 1

UNION ALL

SELECT
    'Ponderado con finalwgt' AS metodo,
    ROUND(SUM(prglngth * finalwgt) / SUM(finalwgt), 3) AS duracion_promedio
FROM embarazos
WHERE outcome = 1""")

Si los resultados son cercanos, eso no vuelve innecesaria la ponderación: indica únicamente que, para esta variable y este resumen, su efecto es pequeño. En otros análisis la diferencia podría ser mayor.

## Criterios para comunicar una conclusión exploratoria

Antes de redactar una conclusión, verificá que:

- identifique con precisión qué grupos y variables se compararon;
- diferencie el resultado numérico de su interpretación;
- valore la diferencia en relación con la variabilidad observada;
- no transforme una asociación en una explicación causal;
- explicite el alcance de los datos y las limitaciones del análisis.

## Ejercicio de cierre

Redactá una conclusión exploratoria de entre tres y cinco oraciones sobre la duración del embarazo en los dos grupos. Incluí un resultado, una interpretación y al menos una limitación.

**Conclusión:** escribí acá tu respuesta.

## Cierre

Una conclusión exploratoria rigurosa explica qué muestran los datos y también qué no permiten afirmar. Aplicá este criterio al completar y revisar la Entrega 4 del Trabajo Práctico Integrador.

**Bibliografía:** Downey, A. B. (2025). *Think Stats* (3.ª ed.), capítulo 1, secciones sobre evidencia e interpretación.

In [ ]:
con.close()